# 13 — Mocny korektor (LanguageTool) vs słaby (edit-distance)

Powtórzenie ablacji z 12 (char/word × TW/GO/CE) z LanguageTool zamiast edit-distance. Korekta z `data/processed/lt/` (`gen_languagetool.py`). Test nietknięty.

In [ ]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score
from thesis_lib import optimal_thresholds as opt_thr
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); plt.rcParams["savefig.dpi"]=300
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
RANDOM_STATE=42
PROCESSED_DIR=Path("../data/processed"); LT=PROCESSED_DIR/"lt"; RESULTS_DIR=Path("../data/results"); FIGURES_DIR=Path("../figures")
DSP={"TW":("twitteremo","tekst")}  # tylko TW (LT scope)
def load(name,col):
    out={}
    for sp in ["train","val","test"]:
        d=pd.read_csv(PROCESSED_DIR/f"{name}_{sp}.csv"); d[col]=d[col].fillna("")
        lt=pd.read_csv(LT/f"{name}_{sp}.csv")["text_lt"].fillna("").tolist()
        out[sp]={"raw":d[col].tolist(),"lt":lt,"y":d[EMOTIONS].values}
    return out
DATA={k:load(*v) for k,v in DSP.items()}
print("załadowano:",list(DATA))

In [ ]:
def vec_for(kind):
    if kind=="char": return TfidfVectorizer(max_features=50_000,ngram_range=(3,5),analyzer="char_wb",min_df=3,sublinear_tf=True,lowercase=True)
    return TfidfVectorizer(max_features=50_000,ngram_range=(1,2),min_df=3,max_df=0.95,sublinear_tf=True,lowercase=True)
def run(tr,va,te,ytr,yval,yte,kind):
    v=vec_for(kind); Xtr=v.fit_transform(tr); Xva=v.transform(va); Xte=v.transform(te)
    clf=OneVsRestClassifier(LogisticRegression(max_iter=1000,C=1.0,class_weight="balanced",solver="liblinear",random_state=RANDOM_STATE))
    clf.fit(Xtr,ytr); thr=opt_thr(yval,clf.predict_proba(Xva))
    return f1_score(yte,(clf.predict_proba(Xte)>=thr).astype(int),average="macro",zero_division=0)

In [3]:
rows=[]
for dom in DATA:
    ytr,yval,yte=DATA[dom]["train"]["y"],DATA[dom]["val"]["y"],DATA[dom]["test"]["y"]
    for kind in ["char","word"]:
        for var in ["raw","lt"]:
            f1=run(DATA[dom]["train"][var],DATA[dom]["val"][var],DATA[dom]["test"][var],ytr,yval,yte,kind)
            rows.append({"zbiór":dom,"model":kind,"wariant":var,"f1_macro":round(f1,3)})
            print(f"{dom} {kind:4s} {var:3s}: {f1:.3f}")
res=pd.DataFrame(rows); res.to_csv(RESULTS_DIR/"correction_lt_ablation.csv",index=False)
piv=res.pivot_table(index=["zbiór","model"],columns="wariant",values="f1_macro")[["raw","lt"]]
piv["Δ_LT"]=(piv["lt"]-piv["raw"]).round(3); display(piv)

TW char raw: 0.474


TW char lt : 0.460


TW word raw: 0.410


TW word lt : 0.404


wariant        raw     lt   Δ_LT
zbiór model                     
TW    char   0.474  0.460 -0.014
      word   0.410  0.404 -0.006

In [4]:
# porównanie ze słabym korektorem (#12)
try:
    weak=pd.read_csv(RESULTS_DIR/"correction_ablation.csv")
    wpv=weak[weak.wariant.isin(["raw","full"])].pivot_table(index=["zbiór","model"],columns="wariant",values="f1_macro")
    wpv["Δ_weak"]=(wpv["full"]-wpv["raw"]).round(3)
    cmp=piv[["Δ_LT"]].join(wpv[["Δ_weak"]])
    cmp.to_csv(RESULTS_DIR/"correction_weak_vs_strong.csv"); display(cmp)


wariant       Δ_LT  Δ_weak
zbiór model               
TW    char  -0.014  -0.012
      word  -0.006   0.009

# Korektor mocny (LT) vs słaby (edit-distance) — wnioski

- TW/char: Δ_weak -0.012 | Δ_LT -0.014
- TW/word: Δ_weak +0.009 | Δ_LT -0.006
